In [1]:
from openai import OpenAI

base_url = "http://localhost:11434/v1"
client = OpenAI(base_url=base_url, api_key='ollama')  # Reads OPENAI_API_KEY from env

# ── Your Python functions ──────────────────────────────────────
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city: {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


flight_durations = {"london": "9 hrs", "paris": "10 hrs", "tokyo": "14 hrs", "berlin": "10 hrs"}

def get_flight_duration(destination_city):
    print(f"Tool called for duration: {destination_city}")
    duration = flight_durations.get(destination_city.lower(), "Unknown duration")
    return f"The flight to {destination_city} takes {duration}"


# ── JSON descriptions (OpenAI format — notice the wrapper) ─────
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

duration_function = {
    "name": "get_flight_duration",
    "description": "Get the flight duration in hours to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the customer wants to fly to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

# ── Pass both tools in OpenAI format ──────────────────────────
tools = [
    {"type": "function", "function": price_function},   # 👈 wrapper here
    {"type": "function", "function": duration_function}
]


# ── Router — runs the right function based on tool name ────────
def run_tool(tool_name, tool_input):
    if tool_name == "get_ticket_price":
        return get_ticket_price(tool_input["destination_city"])
    elif tool_name == "get_flight_duration":
        return get_flight_duration(tool_input["destination_city"])
    else:
        return f"Unknown tool: {tool_name}"


# ── Agent loop ─────────────────────────────────────────────────
import json

def run_agent(user_message):
    print(f"\n👤 You: {user_message}")
    messages = [{"role": "user", "content": user_message}]

    while True:
        response = client.chat.completions.create(
            model="llama3.2:3b",
            messages=messages,
            tools=tools,
            tool_choice="auto"  # let the model decide when to use a tool
        )

        choice = response.choices[0]

        # Claude finished answering
        if choice.finish_reason == "stop":
            print(f"🤖 Agent: {choice.message.content}")
            return choice.message.content

        # Model wants to use a tool
        elif choice.finish_reason == "tool_calls":
            messages.append(choice.message)  # add assistant's tool request to history

            for tool_call in choice.message.tool_calls:
                name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)  # comes back as a JSON string

                print(f"🔧 Using tool: {name}({args})")
                result = run_tool(name, args)
                print(f"   ↳ Result: {result}")

                # Send the result back — OpenAI needs tool_call_id
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,   # 👈 required by OpenAI
                    "content": result
                })


In [2]:
# ── Try it ─────────────────────────────────────────────────────
run_agent("How much is a ticket to Tokyo and how long is the flight?")
# run_agent("What's the cheapest city to fly to?")


👤 You: How much is a ticket to Tokyo and how long is the flight?
🔧 Using tool: get_ticket_price({'destination_city': 'Tokyo'})
Tool called for city: Tokyo
   ↳ Result: The price of a ticket to Tokyo is $1400
🔧 Using tool: get_flight_duration({'destination_city': 'Tokyo'})
Tool called for duration: Tokyo
   ↳ Result: The flight to Tokyo takes 14 hrs
🤖 Agent: Here's the formatted answer:

A round-trip ticket from Tokyo costs approximately $1,400. The total flight duration for this route is around 14 hours.


"Here's the formatted answer:\n\nA round-trip ticket from Tokyo costs approximately $1,400. The total flight duration for this route is around 14 hours."

# Tool Calling Flow

```text
User
 │
 │ "Price to London?"
 ▼

LLM
 │
 │ Decides it needs a tool
 ▼

Assistant Message (role = "assistant")
 │
 └── Tool Call:
     get_ticket_price("London")

 │
 ▼

Python Code Executes the Tool
 │
 └── get_ticket_price("London")
 │
 └── Returns "$799"

 │
 ▼

Tool Message (role = "tool")
 │
 └── "$799"

 │
 ▼

LLM Receives the Tool Result
 │
 ▼

Assistant (Final Response)
 │
 └── "The ticket price to London is $799."
```

## Explanation

1. **User** sends a request.
2. **LLM** determines that it cannot answer directly and decides to call a tool.
3. The LLM returns an **assistant message** containing the tool call (not the final answer).
4. Your Python code receives this tool call and executes the corresponding Python function.
5. The function returns the result (e.g., `$799`).
6. Your code sends this result back to the LLM as a **tool message**.
7. The LLM now has the information it requested and generates the **final natural language response** for the user.